<a href="https://colab.research.google.com/github/run-llama/llama_index/blob/main/docs/examples/agent/gemma4_action_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gemma 4 Agent with Action Tools — Twilio SMS & Bitly Links

This notebook adds **selective action tools** to a local Gemma 4 agent — tools that do something in the world, not just look things up. Two tools are covered:

- **Twilio** — send SMS messages (e.g. daily summaries, alerts to yourself)
- **Bitly** — shorten URLs and generate QR codes for any link the agent produces

**Key safety pattern used throughout:** every action tool is wrapped with a **human-in-the-loop confirmation step** — the agent proposes, you approve, then it executes. The agent never sends a message or creates a link without your explicit `yes`.

**Stack:**
- `gemma4:12b` via Ollama (local LLM)
- `twilio` Python SDK (SMS)
- Bitly MCP server via `llama-index-tools-mcp`

## Prerequisites

**Twilio** (free trial at [twilio.com](https://www.twilio.com)):
- Account SID, Auth Token, and a Twilio phone number from your [Twilio Console](https://console.twilio.com)
- Free trial lets you send to verified numbers

**Bitly** (free at [bitly.com](https://bitly.com)):
- API token from [app.bitly.com/settings/api](https://app.bitly.com/settings/api)
- Free tier includes 1,000 links/month

In [ ]:
!pip install llama-index-llms-ollama llama-index-tools-mcp llama-index-core twilio

In [ ]:
import os

# Twilio credentials — set as environment variables, never hardcode
TWILIO_ACCOUNT_SID = os.environ.get("TWILIO_ACCOUNT_SID", "ACxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx")
TWILIO_AUTH_TOKEN  = os.environ.get("TWILIO_AUTH_TOKEN",  "your_auth_token")
TWILIO_FROM_NUMBER = os.environ.get("TWILIO_FROM_NUMBER", "+15551234567")  # Your Twilio number
MY_PHONE_NUMBER    = os.environ.get("MY_PHONE_NUMBER",    "+15559876543")  # Your personal number

# Bitly credentials
BITLY_TOKEN = os.environ.get("BITLY_TOKEN", "your_bitly_api_token")

## 1. Set up local LLM

In [ ]:
import os

def get_gemma_llm(model: str = "gemma4:12b", timeout: float = 180.0):
    """Auto-selects local Ollama or Ollama Cloud based on OLLAMA_CLOUD_API_KEY.
    Set OLLAMA_CLOUD_API_KEY (from the Ollama app) to run without a local server.
    """
    cloud_key = os.environ.get("OLLAMA_CLOUD_API_KEY", "")
    base_url = os.environ.get("OLLAMA_BASE_URL", "")
    model = os.environ.get("OLLAMA_MODEL", model)
    if cloud_key:
        from llama_index.llms.openai_like import OpenAILike
        print(f"Ollama CLOUD — model: {model}")
        return OpenAILike(
            model=model,
            api_base=base_url or "https://api.ollama.com/v1",
            api_key=cloud_key,
            is_chat_model=True,
            is_function_calling_model=True,
            context_window=128_000,
            timeout=timeout,
        )
    from llama_index.llms.ollama import Ollama
    print(f"LOCAL Ollama — model: {model}")
    return Ollama(model=model, base_url=base_url or "http://localhost:11434", request_timeout=timeout)

llm = get_gemma_llm()

## 2. Human-in-the-loop confirmation wrapper

This pattern gates every action tool behind a confirmation prompt. The agent drafts the action; you type `yes` to execute or anything else to cancel.

In [ ]:
from typing import Callable, Any


def confirm_and_run(action_description: str, action_fn: Callable, **kwargs: Any) -> str:
    """
    Print what the agent wants to do and ask for explicit confirmation
    before executing any real-world action.
    """
    print("\n" + "="*60)
    print("AGENT ACTION PROPOSED:")
    print(action_description)
    print("="*60)
    answer = input("Execute this action? (yes/no): ").strip().lower()
    if answer == "yes":
        return action_fn(**kwargs)
    else:
        return "Action cancelled by user."


print("Confirmation wrapper ready.")

## 3. Twilio SMS tool

Lets the agent send a text message to your phone. Always goes through the confirmation wrapper.

In [ ]:
from twilio.rest import Client as TwilioClient
from llama_index.core.tools import FunctionTool
from typing import Annotated

_twilio_client = TwilioClient(TWILIO_ACCOUNT_SID, TWILIO_AUTH_TOKEN)


def _send_sms_impl(body: str, to: str = MY_PHONE_NUMBER) -> str:
    """Actually send the SMS via Twilio (called only after confirmation)."""
    message = _twilio_client.messages.create(
        body=body,
        from_=TWILIO_FROM_NUMBER,
        to=to,
    )
    return f"SMS sent. SID: {message.sid}"


def send_sms(
    body: Annotated[str, "The SMS message text to send (max 160 characters recommended)"],
    to: Annotated[str, "Recipient phone number in E.164 format, e.g. +15551234567"] = MY_PHONE_NUMBER,
) -> str:
    """
    Send an SMS message via Twilio. Always requires explicit user confirmation
    before sending. Use for summaries, alerts, or reminders to yourself.
    """
    return confirm_and_run(
        action_description=f"Send SMS to {to}:\n\n{body}",
        action_fn=_send_sms_impl,
        body=body,
        to=to,
    )


sms_tool = FunctionTool.from_defaults(fn=send_sms)
print("Twilio SMS tool ready.")

In [ ]:
# Test the tool directly (will prompt for confirmation)
result = sms_tool(body="Hello from your Gemma 4 agent! This is a test message.", to=MY_PHONE_NUMBER)
print(result)

## 4. Bitly link shortener tool

Connects to the Bitly MCP server to shorten URLs and generate QR codes.

In [ ]:
from llama_index.tools.mcp import BasicMCPClient, McpToolSpec

# Connect to Bitly's MCP server via npx
bitly_client = BasicMCPClient(
    "npx",
    args=["-y", "@bitly/mcp-server", "--token", BITLY_TOKEN],
)

bitly_spec = McpToolSpec(
    client=bitly_client,
    # Only expose safe, useful tools
    allowed_tools=[
        "create_short_link",
        "create_qr_code",
        "create_short_link_with_qr",
        "expand",
        "link_clicks_summary",
    ],
)

bitly_tools = await bitly_spec.to_tool_list_async()
print(f"Loaded {len(bitly_tools)} Bitly tools:")
for t in bitly_tools:
    print(f"  • {t.metadata.name}")

In [ ]:
# Quick test — shorten a URL directly
for tool in bitly_tools:
    if tool.metadata.name == "create_short_link":
        result = await tool.acall(long_url="https://docs.llamaindex.ai")
        print(result)
        break

## 5. SMS summary agent

An agent that reads your RAG index (built in the private RAG notebook), drafts a summary, and texts it to you.

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage, Settings
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.tools import QueryEngineTool
from llama_index.core.agent.workflow import ReActAgent

Settings.embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")
Settings.llm = llm

# Load the persisted index from the private RAG notebook
# (or build a fresh one here if you haven't run that notebook)
INDEX_PATH = "./my_rag_index"

try:
    storage_context = StorageContext.from_defaults(persist_dir=INDEX_PATH)
    rag_index = load_index_from_storage(storage_context)
    print("Loaded existing index.")
except Exception:
    # Build a minimal index from scratch for demo purposes
    from llama_index.core import VectorStoreIndex, Document
    sample_doc = Document(text="""
        Daily briefing for June 13, 2026:
        - CapFed checking balance: $4,230.50
        - Meritrust savings: $12,800 (goal: $15,000, gap: $2,200)
        - ETH holding: 0.8 ETH at $3,200 = $2,560
        - BTC holding: 0.02 BTC at $67,000 = $1,340
        - Unread emails: 3 bank notifications, 1 invoice due July 1
    """)
    rag_index = VectorStoreIndex.from_documents([sample_doc])
    print("Built demo index.")

rag_tool = QueryEngineTool.from_defaults(
    query_engine=rag_index.as_query_engine(),
    name="personal_documents",
    description="Search personal documents, financial summaries, emails, and notes.",
)

summary_agent = ReActAgent(
    tools=[rag_tool, sms_tool],
    llm=llm,
    system_prompt=(
        "You are a personal briefing assistant. "
        "When asked for a daily summary, query personal_documents for key info, "
        "draft a concise SMS (under 160 chars), then use send_sms to deliver it. "
        "Always keep SMS text short and actionable."
    ),
    verbose=True,
)
print("Summary agent ready.")

In [ ]:
# Agent drafts and (with your confirmation) sends a daily summary SMS
response = await summary_agent.run(
    f"Send me a daily financial briefing SMS to {MY_PHONE_NUMBER}. "
    "Keep it under 160 characters. Include balance, savings gap, and any urgent items."
)
print(response)

In [ ]:
# Scheduled-style use: generate and send a crypto portfolio SMS
response = await summary_agent.run(
    f"Look up my crypto holdings and text me a brief portfolio summary to {MY_PHONE_NUMBER}."
)
print(response)

## 6. Link shortener + QR agent

An agent that generates content, shortens any URLs it produces, and can text the short link to you.

In [ ]:
link_agent = ReActAgent(
    tools=bitly_tools + [sms_tool],
    llm=llm,
    system_prompt=(
        "You are a link management assistant. "
        "When given a URL, shorten it with Bitly and optionally create a QR code. "
        "If asked to send the link, use send_sms to text it after confirmation."
    ),
    verbose=True,
)

# Shorten a URL
response = await link_agent.run(
    "Shorten this URL and create a QR code for it: "
    "https://github.com/morongosteve/llama_index/pull/72"
)
print(response)

In [ ]:
# Shorten and text to yourself in one shot
response = await link_agent.run(
    f"Shorten https://docs.llamaindex.ai/en/stable/examples/llm/ollama/ "
    f"and text the short link to {MY_PHONE_NUMBER}."
)
print(response)

## 7. Combined agent — RAG + SMS + Bitly

All tools together: the agent can research your documents, shorten links it references, and text you a summary.

In [ ]:
full_agent = ReActAgent(
    tools=[rag_tool, sms_tool] + bitly_tools,
    llm=llm,
    system_prompt=(
        "You are a personal productivity assistant. "
        "You can query personal documents, shorten links, and send SMS summaries. "
        "Always confirm before sending any SMS. Always shorten URLs before including them in messages."
    ),
    verbose=True,
)

response = await full_agent.run(
    f"Look up my savings goal progress, write a motivational SMS update, "
    f"shorten a link to my personal notes, and text everything to {MY_PHONE_NUMBER}."
)
print(response)

## 8. Alert patterns — event-triggered SMS

Combine with a simple scheduler or file watcher to trigger the agent automatically.

In [ ]:
# Pattern: morning briefing function you can call from a cron job or scheduler
async def morning_briefing(phone_number: str) -> None:
    """Generate and (with confirmation) send a morning summary SMS."""
    agent = ReActAgent(
        tools=[rag_tool, sms_tool],
        llm=llm,
        system_prompt="You are a concise morning briefing assistant. Keep SMS under 160 chars.",
    )
    result = await agent.run(
        f"Send a morning briefing SMS to {phone_number} covering: "
        "top financial item, any urgent emails, and one action to take today."
    )
    print(result)


# Uncomment to run:
# await morning_briefing(MY_PHONE_NUMBER)

# To automate, add to crontab:
#   0 8 * * * cd /path/to/project && python -c "import asyncio; from notebook import morning_briefing; asyncio.run(morning_briefing('+15559876543'))"

In [ ]:
# Pattern: threshold alert (e.g. if savings dips below a target)
async def savings_alert(threshold: float = 10000.0) -> None:
    """Send an SMS alert if savings balance drops below threshold."""
    query_engine = rag_index.as_query_engine()
    response = query_engine.query("What is my current savings balance as a number?")

    try:
        balance = float(str(response).replace("$", "").replace(",", "").strip())
        if balance < threshold:
            alert_tool = FunctionTool.from_defaults(fn=send_sms)
            await alert_tool.acall(
                body=f"ALERT: Savings ${balance:,.0f} is below ${threshold:,.0f} target.",
                to=MY_PHONE_NUMBER,
            )
    except ValueError:
        print(f"Couldn't parse balance from: {response}")

# await savings_alert(threshold=13000.0)

## Safety checklist

| Tool | Risk level | Mitigation used |
|---|---|---|
| `send_sms` | Medium — costs money, reaches real phone | `confirm_and_run` wrapper; user types `yes` each time |
| `create_short_link` | Low — creates a public URL | Read the proposed URL before confirming |
| `create_qr_code` | Low — generates an image | No cost; no network side-effects |
| `link_clicks_summary` | None — read-only analytics | Safe |

**Never** give the agent Twilio credentials with MMS or voice call permissions — SMS-only is the minimum scope needed.

## What's next

- **Step 15** — Canva MCP: agent writes copy → Canva builds the design → Bitly shortens the share link → Twilio texts it to you
- **Step 16** — Descript: agent writes a script → Descript edits your video by text
- **Step 25** — Vercel: deploy a web UI so you can trigger briefings from your phone without a terminal